# 🚀 Tier 1 Model Training & Validation (Google Colab GPU)
**Project:** NLP Noise Robustness in Hindi-English Code-Mixed Sentiment Analysis  
**Benchmark:** SemEval-2020 Task 9 (SentiMix Hindi-English)  
**Objective:** Train and validate the 4 Tier 1 model architectures on raw code-mixed text:
1. **Model A:** Word + Character TF-IDF + LinearSVC
2. **Model B:** MuRIL (`google/muril-base-cased`)
3. **Model C:** XLM-RoBERTa (`xlm-roberta-base`)
4. **Model D:** mBERT (`bert-base-multilingual-cased`)

⚠️ **Strict Research Protocol:**
- Training is performed strictly on `data/processed/train.csv` (14,000 samples).
- Checkpoint selection and early stopping are governed strictly by `data/processed/dev.csv` (3,000 samples) Macro-F1.
- `test.csv` is **NEVER** evaluated or loaded in Stage 1.

In [ ]:
# Step 2: Clone Repository (or navigate to workspace if already present)
import os
REPO_DIR = "/content/NLP_Noise_Robustness"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/danishmbox/NLP_Noise_Robustness.git {REPO_DIR}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main
!pwd

In [ ]:
# Step 2: Clone Repository (or set path if already mounted)
import os
if not os.path.exists("NLP_Noise_Robustness"):
    !git clone https://github.com/danishmbox/NLP_Noise_Robustness.git
    %cd NLP_Noise_Robustness
else:
    %cd NLP_Noise_Robustness
    !git pull origin main
!pwd

In [ ]:
# Step 3: Install Required Transformer Acceleration Packages
!pip install -q transformers accelerate datasets evaluate scikit-learn pandas matplotlib seaborn

In [ ]:
# Step 4: Verify Data Integrity and Split Schemas
from pathlib import Path
from src.data.loader import load_train_and_dev_data

train_df, dev_df = load_train_and_dev_data(Path("data/processed"))
print(f"Loaded Train: {len(train_df)} samples | Dev: {len(dev_df)} samples")
print("Train labels:", train_df["sentiment"].value_counts().to_dict())
print("Dev labels:  ", dev_df["sentiment"].value_counts().to_dict())

## 1. Train Model A: Classical Baseline
Word TF-IDF (1, 2) + Char_wb TF-IDF (2, 5) + LinearSVC

In [ ]:
!python src/train.py --model classical

## 2. Train Model B: MuRIL
Indian Language Pre-trained BERT (`google/muril-base-cased`)

In [ ]:
!python src/train.py --model muril

## 3. Train Model C: XLM-RoBERTa
Cross-Lingual RoBERTa (`xlm-roberta-base`)

In [ ]:
!python src/train.py --model xlm_roberta

## 4. Train Model D: mBERT
Canonical Multilingual BERT (`bert-base-multilingual-cased`)

In [ ]:
!python src/train.py --model mbert

## 5. Comparative Development Benchmark Table
Aggregates validation metrics across all executed Tier 1 pipelines.

In [ ]:
import glob
import pandas as pd

summary_files = glob.glob("results/validation/*_val_summary.csv")
if summary_files:
    dfs = [pd.read_csv(f) for f in summary_files]
    benchmark_df = pd.concat(dfs, ignore_index=True)
    benchmark_df = benchmark_df.sort_values(by="macro_f1", ascending=False)
    print("=" * 80)
    print("            TIER 1 VALIDATION BENCHMARK (DEV SET MACRO-F1)            ")
    print("=" * 80)
    display_cols = ["model_name", "macro_f1", "accuracy", "weighted_f1", "neg_f1", "neu_f1", "pos_f1"]
    print(benchmark_df[display_cols].to_string(index=False))
    benchmark_df.to_csv("results/validation/tier1_dev_benchmark.csv", index=False)
else:
    print("No validation summary CSVs found yet.")

## 6. Stage 2B: Robustness Evaluation Under Linguistic Noise (Inference Only)
Evaluates trained model checkpoints on the 8 pre-generated shared noisy dev sets (5%, 10%, 15%, 20% severities)
for both Typographical noise and Hinglish Transliteration variation.

⚠️ **Strict Protocol:**
- Inference only: `model.eval()` with `torch.inference_mode()`.
- Zero retraining: Model weights and representations are untouched.
- Uses identical pre-generated datasets in `results/robustness/datasets/` for 100% fair cross-model comparison.
- `test.csv` is strictly guarded and never evaluated.

### 6.1 Checkpoint Verification & Google Drive Mount
Mounts Google Drive when needed and verifies candidate checkpoint paths.

In [ ]:
import os
from pathlib import Path

# Mount Google Drive if checkpoints reside in Drive
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print('Mounting Google Drive...')
        drive.mount('/content/drive')
    else:
        print('Google Drive already mounted.')
except ImportError:
    print('Running outside Google Colab environment.')

# Verify candidate checkpoint directories
muril_candidates = [
    '/content/drive/MyDrive/NLP_Noise_Robustness_Backup/MuRIL_2026-10-14',
    'results/models/muril_best_checkpoint',
    '/content/NLP_Noise_Robustness/results/models/muril_best_checkpoint',
    '/content/drive/MyDrive/NLP_Noise_Robustness/results/models/muril_best_checkpoint'
]
xlm_candidates = [
    '/content/drive/MyDrive/NLP_Noise_Robustness_Backup/XLM-RoBERTa_2026-10-17',
    'results/models/xlm_roberta_best_checkpoint',
    '/content/NLP_Noise_Robustness/results/models/xlm_roberta_best_checkpoint',
    '/content/drive/MyDrive/NLP_Noise_Robustness/results/models/xlm_roberta_best_checkpoint'
]

def find_valid_checkpoint(name, candidates):
    for c in candidates:
        p = Path(c)
        if p.exists() and (p / 'config.json').exists():
            print(f'[FOUND] {name} checkpoint: {p}')
            return str(p)
    print(f'[NOTE] No pre-verified checkpoint found for {name} among candidates. Will use default results/models/.')
    return None

muril_path = find_valid_checkpoint('MuRIL', muril_candidates)
xlm_path = find_valid_checkpoint('XLM-RoBERTa', xlm_candidates)


### 6.2 Evaluate Model B (MuRIL) Robustness
Runs inference-only evaluation across clean and noisy dev sets using the verified checkpoint.

In [ ]:
# Run MuRIL Robustness Evaluation
muril_arg = f'--model_path "{muril_path}"' if muril_path else ''
!python src/evaluate_robustness.py \
    --model_slug muril \
    --model_name "Model B: MuRIL" \
    --model_type transformer \
    {muril_arg}


### 6.3 Evaluate Model C (XLM-RoBERTa) Robustness
Runs inference-only evaluation across clean and noisy dev sets using the verified checkpoint.

In [ ]:
# Run XLM-RoBERTa Robustness Evaluation
xlm_arg = f'--model_path "{xlm_path}"' if xlm_path else ''
!python src/evaluate_robustness.py \
    --model_slug xlm_roberta \
    --model_name "Model C: XLM-RoBERTa" \
    --model_type transformer \
    {xlm_arg}


### 6.4 Tri-Model Cross-Architecture Robustness Aggregation
Aggregates strictly the three Tier 1 model summaries (`classical_baseline`, `muril`, `xlm_roberta`) into a unified benchmark table
and plots comparative degradation curves.

In [ ]:
# Aggregate strictly the three Tier 1 models into a combined benchmark
!python src/evaluate_robustness.py --aggregate

# Display the benchmark table
import pandas as pd
benchmark_csv = "results/robustness/tier1_cross_model_robustness_benchmark.csv"
if os.path.exists(benchmark_csv):
    df = pd.read_csv(benchmark_csv)
    cols = ["model_name", "noise_type", "severity_pct", "macro_f1", "delta_macro_f1", "retention_pct", "accuracy"]
    print("=" * 95)
    print("          TIER 1 CROSS-ARCHITECTURE ROBUSTNESS BENCHMARK (DEV SET)          ")
    print("=" * 95)
    print(df[cols].to_string(index=False))
else:
    print("Aggregation benchmark file not found.")
